# Video 7: CrewAI with RAG

## What We Will Build
In this notebook, we will build a **Document Research Crew with RAG**. You will learn:
- How RAG (Retrieval-Augmented Generation) works with agents
- How to load and split documents
- How to generate embeddings and store them in a vector database
- How to create a retrieval tool for agents
- How to generate grounded responses with source citations

By the end, you will have an agent that can answer questions based on your documents.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai langchain-community faiss-cpu python-dotenv

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom crewai.tools import toolfrom langchain_openai import ChatOpenAI, OpenAIEmbeddingsfrom langchain_community.vectorstores import FAISSfrom langchain_core.documents import Documentimport osfrom dotenv import load_dotenvload_dotenv()llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY'))embeddings = OpenAIEmbeddings(api_key=os.getenv('OPENAI_API_KEY'))print('Setup complete')

## 3. Theory: RAG for Agents

### What is RAG?
**Retrieval-Augmented Generation** combines:
1. **Retrieval**: Finding relevant documents from a knowledge base
2. **Generation**: Using those documents to ground the LLM's response

### Why RAG for Agents?
- **Accuracy**: Responses are grounded in real documents
- **Freshness**: Knowledge base can be updated without retraining
- **Transparency**: Sources can be cited
- **Domain-specific**: Agents can specialize in your documents

### Architecture
```
Documents -> Split -> Embed -> Store in Vector DB
                                    |
User Query -> Embed -> Similarity Search -> Retrieve Top K
                                    |
                              Combine with Query -> LLM -> Grounded Answer
```

## 4. Load and Prepare Documents

In [ ]:
# Create sample documents (in production, load from files)documents = [    Document(        page_content=(            'LangGraph is a library built on top of LangChain for orchestrating complex agentic workflows. '            'It enables the creation of cyclic graphs, which are essential for agent-like behaviors. '            'LangGraph supports persistence, human-in-the-loop, and streaming out of the box.'        ),        metadata={'source': 'tech_docs', 'topic': 'frameworks'}    ),    Document(        page_content=(            'CrewAI is a framework for orchestrating multi-agent systems. It allows developers to create '            'role-based agents that can collaborate on complex tasks. CrewAI supports sequential and '            'hierarchical processes, built-in memory, and custom tools.'        ),        metadata={'source': 'tech_docs', 'topic': 'frameworks'}    ),    Document(        page_content=(            'Vector databases like FAISS, Pinecone, and Weaviate store high-dimensional embeddings. '            'They enable semantic search by finding vectors that are close to the query vector. '            'FAISS is particularly popular for local development and prototyping.'        ),        metadata={'source': 'tech_docs', 'topic': 'databases'}    ),    Document(        page_content=(            'OpenAI GPT-4 is a large multimodal model that can solve difficult problems with greater '            'accuracy than previous models. It supports text and image inputs and can generate '            'code, essays, and creative content. GPT-4o-mini offers similar capabilities at lower cost.'        ),        metadata={'source': 'tech_docs', 'topic': 'models'}    ),    Document(        page_content=(            'Embeddings are numerical representations of text that capture semantic meaning. '            'OpenAI provides embedding models like text-embedding-3-small and text-embedding-3-large. '            'These models convert text into vectors that can be compared for similarity.'        ),        metadata={'source': 'tech_docs', 'topic': 'embeddings'}    ),]print(f'Loaded {len(documents)} documents')for i, doc in enumerate(documents):    print(f'  {i+1}. {doc.metadata["topic"]}: {doc.page_content[:60]}...')

## 5. Create Vector Store with Embeddings

In [ ]:
# Create FAISS vector store from documentsvector_store = FAISS.from_documents(documents, embeddings)print('Vector store created with FAISS')print(f'  Total vectors: {vector_store.index.ntotal}')# Test similarity searchtest_query = 'What is LangGraph used for?'results = vector_store.similarity_search(test_query, k=2)print(f'\nTest search for: "{test_query}"')print(f'Retrieved {len(results)} documents:')for i, doc in enumerate(results):    print(f'  {i+1}. [{doc.metadata["topic"]}] {doc.page_content[:80]}...')

## 6. Create a Retrieval Tool for the Agent

In [ ]:
@tooldef retrieve_documents(query: str) -> str:    """Search the knowledge base for relevant documents.        Args:        query: The search query to find relevant documents    """    docs = vector_store.similarity_search(query, k=3)        if not docs:        return 'No relevant documents found.'        results = []    for i, doc in enumerate(docs, 1):        results.append(f"Source {i} [{doc.metadata['topic']}]:\n{doc.page_content}")        return '\n\n---\n\n'.join(results)print('Retrieval tool created')print(f'  Name: {retrieve_documents.name}')print(f'  Description: {retrieve_documents.description}')

## 7. Create the RAG-Enabled Agent

In [ ]:
# Create agent with retrieval toolrag_agent = Agent(    role='Document Research Specialist',    goal='Answer questions accurately using only the provided document knowledge base',    backstory=(        'You are a meticulous research specialist who always bases answers on retrieved documents. '        'You never make up information. If the documents do not contain the answer, you say so. '        'You cite your sources clearly.'    ),    llm=llm,    tools=[retrieve_documents],    verbose=True,    allow_delegation=False)print('RAG-enabled agent created')

## 8. Create RAG Tasks

In [ ]:
# Task 1: Answer about frameworksframework_task = Task(    description=(        'Answer the question: What are the key features of LangGraph and how does it compare to CrewAI? '        'Use the retrieve_documents tool to find relevant information. Cite your sources.'    ),    expected_output=(        'An accurate answer based on retrieved documents with clear source citations. '        'If information is not in the documents, state that clearly.'    ),    agent=rag_agent)# Task 2: Answer about embeddingsembedding_task = Task(    description=(        'Answer the question: What are embeddings and which embedding models does OpenAI provide? '        'Use the retrieve_documents tool to find relevant information.'    ),    expected_output='An accurate answer about embeddings based on retrieved documents.',    agent=rag_agent)print('RAG tasks created')

## 9. Build and Execute the RAG Crew

In [ ]:
rag_crew = Crew(    agents=[rag_agent],    tasks=[framework_task, embedding_task],    process=Process.sequential,    verbose=True)print('\n' + '=' * 60)print('EXECUTING RAG DOCUMENT RESEARCH CREW')print('=' * 60)rag_result = rag_crew.kickoff()print('\n' + '=' * 60)print('RAG CREW OUTPUT')print('=' * 60)print(rag_result)

## 10. Test with Out-of-Scope Query

Let's see how the agent handles a question not in the knowledge base.

In [ ]:
# Task with out-of-scope queryout_of_scope_task = Task(    description=(        'Answer the question: What is quantum computing? '        'Use the retrieve_documents tool to find relevant information.'    ),    expected_output='An honest response if the information is not available in documents.',    agent=rag_agent)oos_crew = Crew(    agents=[rag_agent],    tasks=[out_of_scope_task],    process=Process.sequential,    verbose=True)print('\n' + '=' * 60)print('TESTING OUT-OF-SCOPE QUERY')print('=' * 60)oos_result = oos_crew.kickoff()print('\nOutput:')print(oos_result)

## 11. RAG Best Practices

1. **Chunk size**: Balance between context and relevance
2. **Overlap**: Use overlapping chunks to preserve context
3. **Top-K**: Retrieve enough documents but not too many
4. **Re-ranking**: Use a re-ranker for better retrieval quality
5. **Source citation**: Always cite sources for transparency
6. **Hallucination guard**: Instruct agents to say 'I don't know' when documents don't contain the answer

---

## ✅ Summary

You have successfully:
1. ✅ Loaded and prepared documents for RAG
2. ✅ Generated embeddings and stored them in FAISS
3. ✅ Created a retrieval tool for semantic search
4. ✅ Built a RAG-enabled agent with document access
5. ✅ Executed tasks that retrieve and cite documents
6. ✅ Tested out-of-scope query handling

In the next notebook, we will explore **Multi-Agent Collaboration**.